# Assignment — Linear Regression on `penguins.csv`

**Question:** given a penguin's measurements, how heavy is it?

Target: `body_mass_g` &nbsp;·&nbsp; File: `penguins.csv`

Use only the numeric columns: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`.

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv('/Users/ankit/BOX 1/B1 ROOM 1/AI&ML/class11/penguins.csv')
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


---
## 1. Look at the data

> **Flow:** Shape, columns, missing values.

Run `.info()`. Which columns have missing values, and how many?

In [3]:
df.info()

print()
print("Shape:", df.shape)
print()
print("Missing values per column:")
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB

Shape: (344, 7)

Missing values per column:


species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

---
## 2. Clean

> **Flow:** Keep the four numeric columns, drop the rows with blanks.

We cannot fill in `body_mass_g` — that is the answer we are trying to predict.
Rows missing it have to go.

Keep `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`,
then `dropna()`.

In [4]:
df_clean = df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']].dropna()

print("Rows before:", len(df))
print("Rows after :", len(df_clean))
df_clean.head()

Rows before: 344
Rows after : 342


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,39.1,18.7,181.0,3750.0
1,39.5,17.4,186.0,3800.0
2,40.3,18.0,195.0,3250.0
4,36.7,19.3,193.0,3450.0
5,39.3,20.6,190.0,3650.0


*How many rows are left?*

**342 rows** are left (344 − 2). The same 2 rows were blank in all four numeric columns.

---
## 3. Features and target

In [5]:
X = df_clean[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]
y = df_clean['body_mass_g']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (342, 3)
y shape: (342,)


---
## 4. Split

> **Flow:** `test_size=0.2`, `random_state=42`.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (273, 3) (273,)
Test : (69, 3) (69,)


---
## 5. One feature

> **Flow:** Start with `flipper_length_mm` alone.

Train `LinearRegression`. Print the coefficient, the intercept and the R².

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model_1 = LinearRegression()
model_1.fit(X_train[['flipper_length_mm']], y_train)

y_pred_1 = model_1.predict(X_test[['flipper_length_mm']])

print("Coefficient:", model_1.coef_[0])
print("Intercept  :", model_1.intercept_)
print("R²         :", r2_score(y_test, y_pred_1))

Coefficient: 48.829682818993014
Intercept  : -5614.067120606901
R²         : 0.7820354165340795


*Coefficient:* **48.83** &nbsp;&nbsp; *R²:* **0.782**

**Q.** The coefficient is about 48. Write one line explaining what that means in plain words —
what happens to the predicted weight if a penguin's flipper is 1 mm longer?

**A.** For every 1 mm longer flipper, the model predicts the penguin is about **49 g heavier**.

---
## 6. All three features

> **Flow:** Same model, two more columns.

Print the R².

In [8]:
model_3 = LinearRegression()
model_3.fit(X_train, y_train)

y_pred_3 = model_3.predict(X_test)

print("R² with flipper only:", r2_score(y_test, y_pred_1))
print("R² with all three   :", r2_score(y_test, y_pred_3))

R² with flipper only: 0.7820354165340795
R² with all three   : 0.7877806019338434


*R² with flipper only:* **0.782** &nbsp;&nbsp; *R² with all three:* **0.788**

**Q.** The score barely moved. That is not a mistake — it is the interesting part of this
assignment. Write two lines on why adding `bill_length_mm` and `bill_depth_mm` gave almost
nothing.

*Hint: what do all three columns really measure?*

**A.** All three columns mostly measure the same thing: **how big the penguin is**. A big penguin has long flippers *and* a long bill
(bill length vs flipper length correlation ≈ 0.66). Flipper length (correlation 0.87 with mass) already captures that size, so the bill columns add almost no new information.

---
## 8. Metrics

> **Flow:** MAE, RMSE, R².

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y_test, y_pred_3)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_3))
r2 = r2_score(y_test, y_pred_3)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)
print()
print("Average body mass:", y.mean())
print("MAE as % of average mass:", mae / y.mean() * 100)

MAE : 310.5474404912674
RMSE: 375.6441343110776
R²  : 0.7877806019338434

Average body mass: 4201.754385964912
MAE as % of average mass: 7.390899418790081


*MAE:* **310.5 g** &nbsp;&nbsp; *RMSE:* **375.6 g** &nbsp;&nbsp; *R²:* **0.788**

**Q.** The average penguin weighs about 4200 g. Is your MAE large or small compared to that?
One line.

**A.** Fairly **small**: 310 g is about **7%** of 4200 g, so a typical prediction is off by less than a tenth of the penguin's weight.

---
## 8. Coefficients

In [10]:
# Print each feature name next to its coefficient
for name, coef in zip(X.columns, model_3.coef_):
    print(f"{name:20s} {coef:8.2f}")

print(f"{'intercept':20s} {model_3.intercept_:8.2f}")

bill_length_mm           4.01
bill_depth_mm           10.92
flipper_length_mm       48.67
intercept            -5946.04


---
## 9. Questions

**Q1.** Why can we not use `accuracy_score` on this problem?

**Q2.** We dropped rows where `body_mass_g` was missing instead of filling them with the
median. Why is filling in the target a bad idea?

**Q3.** In class, adding more features to the mpg model raised R² from 0.723 to 0.824.
Here it barely changed. In two lines — what is different about these two datasets?